# 03 Doc report train tren Google Colab

Notebook nay chi de mo report, loss, confusion matrix va so sanh nhanh run da train trong Google Drive.

## 1. Mount Google Drive

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image as IPImage, display
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/canteen_checkout')
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs'
print('DRIVE_RUNS_DIR =', DRIVE_RUNS_DIR)


## 2. Chon run moi nhat hoac sua `RUN_PATH` thu cong

In [ ]:
runs = sorted([p for p in DRIVE_RUNS_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
if not runs:
    raise FileNotFoundError("Khong co run nao trong MyDrive/canteen_checkout/runs")

for idx, run_path in enumerate(runs[:10], 1):
    print(idx, run_path.name)

RUN_PATH = runs[0]
REPORT_OUT = RUN_PATH / "outputs" / "reports"
MODEL_PATH = RUN_PATH / "models" / "dish_classifier.pt"
print("RUN_PATH =", RUN_PATH)
print("REPORT_OUT =", REPORT_OUT)
print("MODEL_PATH =", MODEL_PATH)


## 3. Classification report

In [ ]:
report_file = REPORT_OUT / "classification_report.txt"
print(report_file.read_text(encoding="utf-8"))


## 4. Training history bang va bieu do

In [ ]:
history_file = REPORT_OUT / "training_history.json"
history = json.loads(history_file.read_text(encoding="utf-8"))
df = pd.DataFrame(history)
display(df)

history_png = REPORT_OUT / "training_history.png"
if history_png.exists():
    display(IPImage(filename=str(history_png), width=900))


## 5. Confusion matrix

In [ ]:
cm_path = REPORT_OUT / "confusion_matrix.png"
if cm_path.exists():
    display(IPImage(filename=str(cm_path), width=900))
else:
    print("Khong thay confusion_matrix.png")


## 6. Ghi chu nhanh de doc overfitting

- Train loss giam, val loss cung giam: tot.
- Train loss giam nhung val loss tang: co dau hieu overfitting.
- Train acc rat cao nhung val acc dung im/giam: dataset hoac augmentation can xem lai.
- Macro F1 thap hon weighted F1 nhieu: class it anh dang yeu.

## 7. YOLO detector report

Dung cell nay sau khi chay notebook `04_colab_train_yolo_detector.ipynb`. YOLO thuong ghi `results.csv`, confusion matrix, PR curve trong thu muc `detector/` cua run.

In [ ]:
detector_dirs = list(RUN_PATH.rglob("detector")) + list(RUN_PATH.rglob("egg_fish_detector"))
if not detector_dirs:
    print("Khong thay thu muc detector trong run nay. Neu vua train YOLO, hay sua RUN_PATH sang run detector.")
else:
    DETECTOR_REPORT_DIR = detector_dirs[0]
    print("DETECTOR_REPORT_DIR =", DETECTOR_REPORT_DIR)
    results_csv = next(DETECTOR_REPORT_DIR.rglob("results.csv"), None)
    if results_csv:
        display(pd.read_csv(results_csv).tail())
    for name in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
        found = next(DETECTOR_REPORT_DIR.rglob(name), None)
        if found:
            print(name, found)
            display(IPImage(filename=str(found), width=900))